# Unrevised-Data Check for Market Macro Series

Compare current FRED values with ALFRED initial-release records for the retained daily market inputs:
`T10Y2Y`, `DGS3MO`, `BAA10Y`, `T5YIE`, `T10YIE`, `VIXCLS`, and `DCOILWTICO`.


$$\text{revision}_t = x_t^{\text{current FRED}} - x_t^{\text{ALFRED first release}}$$

The comparison shows revisions and the period with matching FRED and ALFRED records.

Set `FRED_API_KEY` in the environment or a repository-level `.env` file. For a file outside this checkout, set `FRED_ENV_FILE` to its path before running. Keys are never displayed or saved in outputs.


In [1]:
import datetime as dt
import os
import shlex
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    import truststore
    truststore.inject_into_ssl()
except ImportError:
    pass

FRED_OBSERVATIONS_URL = "https://api.stlouisfed.org/fred/series/observations"
FRED_SERIES_URL = "https://api.stlouisfed.org/fred/series"

START = "2000-01-01"
END = "2026-07-31"
SERIES_IDS = [
    "T10Y2Y",
    "DGS3MO",
    "BAA10Y",
    "T5YIE",
    "T10YIE",
    "VIXCLS",
    "DCOILWTICO",
]

PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "dataset/macro_variables/fetch_fred_data.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run this notebook from within the project checkout.")


print(f"Project root: {PROJECT_ROOT}")


Project root: /Users/linru/Desktop/dse4101/dse4101


## API Helpers

`output_type=4` returns initial-release observations. The ALFRED call is chunked by real-time year because long daily-series vintage windows can exceed the FRED JSON API vintage-date limit.

In [2]:
def load_dotenv_fallback(path=None):
    """Minimal .env loader so FRED_API_KEY can live in the repo-level .env file."""
    if path is None:
        path = Path(os.getenv("FRED_ENV_FILE", PROJECT_ROOT / ".env"))
    path = Path(path)
    if not path.exists():
        return
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.removeprefix("export ").partition("=")
        if key.strip() != "FRED_API_KEY":
            continue
        tokens = shlex.split(value, comments=True)
        if tokens:
            os.environ.setdefault("FRED_API_KEY", tokens[0])


def get_fred_api_key():
    load_dotenv_fallback()
    api_key = os.getenv("FRED_API_KEY")
    if not api_key:
        raise RuntimeError("Missing FRED_API_KEY. Set it in your shell or repo-level .env before running this notebook.")
    return api_key


def verify_series_exists(series_id, api_key):
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
    }
    try:
        response = requests.get(FRED_SERIES_URL, params=params, timeout=30)
    except requests.RequestException as exc:
        raise RuntimeError(f"Series lookup failed: {type(exc).__name__}") from None
    try:
        payload = response.json()
    except ValueError:
        raise RuntimeError(f"FRED returned non-JSON data (HTTP {response.status_code}).") from None
    if response.status_code >= 400 or "error_message" in payload:
        raise RuntimeError(
            f"{series_id} does not exist in current FRED: "
            f"{str(payload.get('error_message', response.text)).replace(api_key, '[REDACTED]')}"
        )
    return True


def parse_observations(payload):
    df = pd.DataFrame(payload.get("observations", []))
    if df.empty:
        return pd.DataFrame(columns=["date", "realtime_start", "realtime_end", "value"])
    df["date"] = pd.to_datetime(df["date"])
    if "realtime_start" in df.columns:
        df["realtime_start"] = pd.to_datetime(df["realtime_start"])
    if "realtime_end" in df.columns:
        df["realtime_end"] = pd.to_datetime(df["realtime_end"])
    df["value"] = pd.to_numeric(df["value"].replace(".", pd.NA), errors="coerce")
    return df


def fred_get(params, timeout=30):
    try:
        response = requests.get(FRED_OBSERVATIONS_URL, params=params, timeout=timeout)
    except requests.RequestException as exc:
        raise RuntimeError(f"Observation request failed: {type(exc).__name__}") from None
    try:
        payload = response.json()
    except ValueError:
        raise RuntimeError(f"FRED returned non-JSON data (HTTP {response.status_code}).") from None
    if response.status_code >= 400 or "error_message" in payload:
        message = payload.get("error_message", response.text)
        raise RuntimeError(str(message).replace(str(params.get("api_key", "")), "[REDACTED]"))
    return payload


def is_empty_vintage_window_error(exc):
    message = str(exc).lower()
    return (
        "no vintage dates exist" in message
        or "no observations" in message
        or "series does not exist" in message
    )


def yearly_chunks(start, end):
    start_dt = pd.Timestamp(start)
    end_dt = pd.Timestamp(end)
    chunk_start = start_dt
    while chunk_start <= end_dt:
        chunk_end = min(pd.Timestamp(year=chunk_start.year, month=12, day=31), end_dt)
        yield chunk_start.strftime("%Y-%m-%d"), chunk_end.strftime("%Y-%m-%d")
        chunk_start = chunk_end + pd.Timedelta(days=1)


def fetch_alfred_first_release(series_id, api_key, start=START, end=END):
    # First prove the series ID is valid in current FRED. After this check,
    # historical ALFRED "series does not exist" errors mean unavailable in that vintage window.
    verify_series_exists(series_id, api_key)

    frames = []
    for rt_start, rt_end in yearly_chunks(start, end):
        params = {
            "api_key": api_key,
            "file_type": "json",
            "series_id": series_id,
            "output_type": 4,
            "realtime_start": rt_start,
            "realtime_end": rt_end,
            "observation_start": start,
            "observation_end": end,
            "sort_order": "asc",
        }
        try:
            payload = fred_get(params)
        except RuntimeError as exc:
            if is_empty_vintage_window_error(exc):
                continue
            raise
        frame = parse_observations(payload)
        if not frame.empty:
            frames.append(frame)
    if not frames:
        return pd.DataFrame(columns=["date", "realtime_start", "realtime_end", "value"])
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=["date"]).sort_values(["date", "realtime_start"])
    # Chunk boundaries can duplicate records; keep one first-release row per observation date.
    df = df.drop_duplicates(subset=["date"], keep="first")
    return df.reset_index(drop=True)


def fetch_current_fred(series_id, api_key, start=START, end=END):
    params = {
        "api_key": api_key,
        "file_type": "json",
        "series_id": series_id,
        "observation_start": start,
        "observation_end": end,
        "sort_order": "asc",
    }
    payload = fred_get(params)
    return parse_observations(payload).sort_values("date").reset_index(drop=True)

## Comparison Functions

In [3]:
def compare_first_release_to_current(series_id, api_key, tolerance=0.0):
    first = fetch_alfred_first_release(series_id, api_key)
    current = fetch_current_fred(series_id, api_key)

    first = first.rename(columns={"value": "first_release_value"})
    current = current.rename(columns={"value": "current_value"})[["date", "current_value"]]

    compare = first[["date", "realtime_start", "first_release_value"]].merge(
        current, on="date", how="outer", validate="one_to_one"
    )
    has_first = compare["first_release_value"].notna()
    has_current = compare["current_value"].notna()
    n_current = int(has_current.sum())
    n_current_without_first = int((has_current & ~has_first).sum())
    n_first_without_current = int((has_first & ~has_current).sum())
    compare = compare.loc[has_first & has_current].copy()
    compare["revision"] = compare["current_value"] - compare["first_release_value"]
    compare["abs_revision"] = compare["revision"].abs()
    compare["changed"] = compare["abs_revision"] > tolerance

    compare["publication_lag_days"] = (compare["realtime_start"] - compare["date"]).dt.days
    compare["publication_lag_present"] = compare["publication_lag_days"] > 0

    summary = {
        "series": series_id,
        "n_current_observations": n_current,
        "n_observations_checked": int(len(compare)),
        "n_current_without_first_release": n_current_without_first,
        "n_first_release_without_current": n_first_without_current,
        "full_archive_coverage": bool(n_current and not n_current_without_first and not n_first_without_current),
        "n_revised_observations": int(compare["changed"].sum()),
        "max_abs_revision": float(compare["abs_revision"].max()) if len(compare) else np.nan,
        "mean_abs_revision": float(compare["abs_revision"].mean()) if len(compare) else np.nan,
        "publication_lag_present": bool(compare["publication_lag_present"].any()) if len(compare) else pd.NA,
        "max_publication_lag_days": float(compare["publication_lag_days"].max()) if len(compare) else np.nan,
        "median_publication_lag_days": float(compare["publication_lag_days"].median()) if len(compare) else np.nan,
        "first_comparable_date": compare["date"].min() if len(compare) else pd.NaT,
        "last_comparable_date": compare["date"].max() if len(compare) else pd.NaT,
        "treatment": (
            "Revisions detected; use or investigate point-in-time values"
            if compare["changed"].any() else
            "No revisions on compared dates; archive coverage incomplete"
            if n_current_without_first or n_first_without_current else
            "No revisions on compared dates; publication timing still requires review"
        ),
        "status": "OK" if len(compare) else "NO_COMPARABLE_OBSERVATIONS",
    }
    return summary, compare


def run_unrevised_check(series_ids=SERIES_IDS):
    api_key = get_fred_api_key()
    summaries = []
    comparisons = {}

    for series_id in series_ids:
        print(f"Checking {series_id} ...")
        try:
            summary, compare = compare_first_release_to_current(series_id, api_key)
            comparisons[series_id] = compare
        except Exception as exc:
            summary = {
                "series": series_id,
                "n_current_observations": pd.NA,
                "n_observations_checked": 0,
                "n_current_without_first_release": pd.NA,
                "n_first_release_without_current": pd.NA,
                "full_archive_coverage": False,
                "n_revised_observations": pd.NA,
                "max_abs_revision": np.nan,
                "mean_abs_revision": np.nan,
                "publication_lag_present": pd.NA,
                "max_publication_lag_days": np.nan,
                "median_publication_lag_days": np.nan,
                "first_comparable_date": pd.NaT,
                "last_comparable_date": pd.NaT,
                "treatment": "Audit failed; inspect API/source availability",
                "status": f"ERROR: {exc}",
            }
            comparisons[series_id] = pd.DataFrame()
        summaries.append(summary)

    summary_df = pd.DataFrame(summaries)
    return summary_df, comparisons

## Run check

`status = OK` means the requests and comparison succeeded. It does not mean the series is unrevised or safe for same-day use. Coverage counts include non-missing current observations for which no comparable initial-release value was returned.


In [4]:
summary_df, comparisons = run_unrevised_check()
summary_df

Checking T10Y2Y ...
Checking DGS3MO ...
Checking BAA10Y ...
Checking T5YIE ...
Checking T10YIE ...
Checking VIXCLS ...
Checking DCOILWTICO ...


,series,n_current_observations,n_observations_checked,n_current_without_first_release,n_first_release_without_current,full_archive_coverage,n_revised_observations,max_abs_revision,mean_abs_revision,publication_lag_present,max_publication_lag_days,median_publication_lag_days,first_comparable_date,last_comparable_date,treatment,status
0,T10Y2Y,6648,3130,3518,0,False,2,0.01,0.000006,True,6.0,0.0,2014-01-27,2026-07-31,Revisions detected; use or investigate point-i...,OK
1,DGS3MO,6648,5273,1375,0,False,1,0.01,0.000002,True,9.0,1.0,2005-06-27,2026-07-30,Revisions detected; use or investigate point-i...,OK
2,BAA10Y,6642,3123,3519,0,False,13,0.21,0.000256,True,166.0,1.0,2014-01-27,2026-07-30,Revisions detected; use or investigate point-i...,OK
3,T5YIE,5899,3130,2769,0,False,2,0.02,0.000010,True,6.0,0.0,2014-01-27,2026-07-31,Revisions detected; use or investigate point-i...,OK
4,T10YIE,5899,3130,2769,0,False,2,0.01,0.000006,True,6.0,0.0,2014-01-27,2026-07-31,Revisions detected; use or investigate point-i...,OK
5,VIXCLS,6717,3975,2742,0,False,0,0.00,0.000000,True,5.0,0.0,2010-11-22,2026-07-31,No revisions on compared dates; archive covera...,OK
6,DCOILWTICO,6286,3761,2525,76,False,27,6.48,0.008918,True,15.0,6.0,2011-04-06,2026-07-27,Revisions detected; use or investigate point-i...,OK


## Value revisions, archive coverage, and timing

The value test applies only to comparable dates. Full archive coverage is reported separately. Positive ALFRED vintage lags require review against actual publication schedules; zero lag alone is not proof of availability before the model's trading cutoff.


In [5]:
from IPython.display import display

pass_fail = summary_df.copy()
pass_fail["passes_unrevised_value_test"] = (
    (pass_fail["status"] == "OK")
    & (pass_fail["n_observations_checked"] > 0)
    & (pass_fail["n_revised_observations"] == 0)
    & (pass_fail["max_abs_revision"].fillna(np.inf) == 0)
)
display(pass_fail[[
    "series", "n_current_observations", "n_observations_checked",
    "n_revised_observations", "max_abs_revision",
    "n_current_without_first_release", "n_first_release_without_current",
    "passes_unrevised_value_test", "full_archive_coverage", "status",
]])
display(pass_fail[[
    "series", "first_comparable_date", "last_comparable_date",
    "median_publication_lag_days", "max_publication_lag_days", "treatment",
]])


,series,n_current_observations,n_observations_checked,n_revised_observations,max_abs_revision,n_current_without_first_release,n_first_release_without_current,passes_unrevised_value_test,full_archive_coverage,status
0,T10Y2Y,6648,3130,2,0.01,3518,0,False,False,OK
1,DGS3MO,6648,5273,1,0.01,1375,0,False,False,OK
2,BAA10Y,6642,3123,13,0.21,3519,0,False,False,OK
3,T5YIE,5899,3130,2,0.02,2769,0,False,False,OK
4,T10YIE,5899,3130,2,0.01,2769,0,False,False,OK
5,VIXCLS,6717,3975,0,0.00,2742,0,True,False,OK
6,DCOILWTICO,6286,3761,27,6.48,2525,76,False,False,OK


,series,first_comparable_date,last_comparable_date,median_publication_lag_days,max_publication_lag_days,treatment
0,T10Y2Y,2014-01-27,2026-07-31,0.0,6.0,Revisions detected; use or investigate point-i...
1,DGS3MO,2005-06-27,2026-07-30,1.0,9.0,Revisions detected; use or investigate point-i...
2,BAA10Y,2014-01-27,2026-07-30,1.0,166.0,Revisions detected; use or investigate point-i...
3,T5YIE,2014-01-27,2026-07-31,0.0,6.0,Revisions detected; use or investigate point-i...
4,T10YIE,2014-01-27,2026-07-31,0.0,6.0,Revisions detected; use or investigate point-i...
5,VIXCLS,2010-11-22,2026-07-31,0.0,5.0,No revisions on compared dates; archive covera...
6,DCOILWTICO,2011-04-06,2026-07-27,6.0,15.0,Revisions detected; use or investigate point-i...


## Largest Revisions, If Any

In [6]:
for series_id, compare in comparisons.items():
    if compare.empty or not compare["changed"].any():
        continue
    print(f"\n{series_id}: largest absolute revisions")
    display(
        compare.sort_values("abs_revision", ascending=False)[[
            "date",
            "realtime_start",
            "first_release_value",
            "current_value",
            "revision",
            "abs_revision",
            "publication_lag_days",
        ]].head(20)
    )


T10Y2Y: largest absolute revisions

DGS3MO: largest absolute revisions

BAA10Y: largest absolute revisions

T5YIE: largest absolute revisions

T10YIE: largest absolute revisions

DCOILWTICO: largest absolute revisions


,date,realtime_start,first_release_value,current_value,revision,abs_revision,publication_lag_days
5804,2022-04-01,2022-04-01,-0.06,-0.05,0.01,0.01,0
5837,2022-05-18,2022-05-18,0.20,0.21,0.01,0.01,0
3670,2014-01-27,2014-01-28,2.41,2.41,0.00,0.00,1
5848,2022-06-02,2022-06-02,0.27,0.27,0.00,0.00,0
5838,2022-05-19,2022-05-19,0.21,0.21,0.00,0.00,0
5839,2022-05-20,2022-05-20,0.18,0.18,0.00,0.00,0
5840,2022-05-23,2022-05-23,0.21,0.21,0.00,0.00,0
5841,2022-05-24,2022-05-24,0.26,0.26,0.00,0.00,0
5842,2022-05-25,2022-05-25,0.27,0.27,0.00,0.00,0
5843,2022-05-26,2022-05-26,0.29,0.29,0.00,0.00,0


,date,realtime_start,first_release_value,current_value,revision,abs_revision,publication_lag_days
2193,2008-05-29,2008-06-02,1.91,1.90,-0.01,0.01,4
1430,2005-06-27,2005-07-06,3.15,3.15,0.00,0.00,9
5104,2019-07-26,2019-07-29,2.12,2.12,0.00,0.00,3
5103,2019-07-25,2019-07-26,2.10,2.10,0.00,0.00,1
5102,2019-07-24,2019-07-25,2.10,2.10,0.00,0.00,1
5101,2019-07-23,2019-07-24,2.06,2.06,0.00,0.00,1
5100,2019-07-22,2019-07-23,2.09,2.09,0.00,0.00,1
5099,2019-07-19,2019-07-22,2.06,2.06,0.00,0.00,3
5098,2019-07-18,2019-07-19,2.05,2.05,0.00,0.00,1
5097,2019-07-17,2019-07-18,2.14,2.14,0.00,0.00,1


,date,realtime_start,first_release_value,current_value,revision,abs_revision,publication_lag_days
5978,2022-12-01,2022-12-02,2.30,2.09,-0.21,0.21,1
6533,2025-01-16,2025-01-17,1.61,1.46,-0.15,0.15,1
6532,2025-01-15,2025-01-16,1.56,1.44,-0.12,0.12,1
5330,2020-06-08,2020-06-09,2.88,2.81,-0.07,0.07,1
5329,2020-06-05,2020-06-08,2.91,2.86,-0.05,0.05,3
6497,2024-11-27,2024-12-02,1.48,1.43,-0.05,0.05,5
5324,2020-05-29,2020-06-01,3.11,3.08,-0.03,0.03,3
5326,2020-06-02,2020-06-03,3.11,3.08,-0.03,0.03,1
5321,2020-05-26,2020-05-27,3.12,3.14,0.02,0.02,1
5327,2020-06-03,2020-06-04,3.04,3.02,-0.02,0.02,1


,date,realtime_start,first_release_value,current_value,revision,abs_revision,publication_lag_days
4482,2020-03-09,2020-03-10,0.78,0.80,0.02,0.02,1
5021,2022-04-01,2022-04-01,3.29,3.28,-0.01,0.01,0
2887,2014-01-27,2014-01-28,1.72,1.72,0.00,0.00,1
5066,2022-06-03,2022-06-03,3.04,3.04,0.00,0.00,0
5056,2022-05-20,2022-05-20,2.88,2.88,0.00,0.00,0
5057,2022-05-23,2022-05-23,2.92,2.92,0.00,0.00,0
5058,2022-05-24,2022-05-24,2.87,2.87,0.00,0.00,0
5059,2022-05-25,2022-05-25,2.88,2.88,0.00,0.00,0
5060,2022-05-26,2022-05-26,2.96,2.96,0.00,0.00,0
5061,2022-05-27,2022-05-27,2.95,2.95,0.00,0.00,0


,date,realtime_start,first_release_value,current_value,revision,abs_revision,publication_lag_days
5054,2022-05-18,2022-05-18,2.71,2.72,0.01,0.01,0
5021,2022-04-01,2022-04-01,2.79,2.80,0.01,0.01,0
2887,2014-01-27,2014-01-28,2.14,2.14,0.00,0.00,1
5065,2022-06-02,2022-06-02,2.66,2.66,0.00,0.00,0
5055,2022-05-19,2022-05-19,2.59,2.59,0.00,0.00,0
5056,2022-05-20,2022-05-20,2.55,2.55,0.00,0.00,0
5057,2022-05-23,2022-05-23,2.60,2.60,0.00,0.00,0
5058,2022-05-24,2022-05-24,2.55,2.55,0.00,0.00,0
5059,2022-05-25,2022-05-25,2.56,2.56,0.00,0.00,0
5060,2022-05-26,2022-05-26,2.62,2.62,0.00,0.00,0


,date,realtime_start,first_release_value,current_value,revision,abs_revision,publication_lag_days
5272,2020-03-18,2020-03-25,26.96,20.48,-6.48,6.48,7
5274,2020-03-20,2020-03-25,25.09,19.48,-5.61,5.61,5
5273,2020-03-19,2020-03-25,20.48,25.09,4.61,4.61,6
5275,2020-03-23,2020-03-25,19.48,23.33,3.85,3.85,2
3064,2011-09-30,2011-10-05,81.87,78.93,-2.94,2.94,5
3044,2011-09-02,2011-09-08,88.93,86.45,-2.48,2.48,6
5670,2021-09-27,2021-09-29,73.43,75.54,2.11,2.11,2
3049,2011-09-09,2011-09-14,89.05,87.24,-1.81,1.81,5
3054,2011-09-16,2011-09-21,89.40,87.96,-1.44,1.44,5
3109,2011-12-02,2011-12-07,100.20,100.97,0.77,0.77,5




API definitions: [FRED observations](https://fred.stlouisfed.org/docs/api/fred/series_observations.html), where `output_type=4` requests initial-release observations.
